# Models Demo

This demo shows how the models considered for the project architecture act on the raw dataset

In [ ]:
import os
import torch
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import DataLoader
import cv2
import numpy as np
from models import utils
import sys
from sixdrepnet import SixDRepNet
from dataset import LP300W

In [4]:
if torch.cuda.is_available():
    device_name = "cuda"
elif torch.backends.mps.is_available():
    device_name = "mps"
else:
    device_name = "cpu"
    
device = torch.device(device_name)
print(f"Code runs in {device}")

Code runs in cuda


## Load data

In [ ]:
dataset = LP300W(
    root="dataset/300W_LP",
    transform=T.ToTensor()
)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

img, pose = next(iter(loader))
print(img.shape, pose.shape)


torch.Size([32, 3, 450, 450]) torch.Size([32, 3])


## Model1 (TokenHPE)

In [11]:
sys.path.append("./models")

from token_hpe import TokenHPE

model = TokenHPE(depth=3)
checkpoint = torch.load("./weights/TokenHPEv1-ViTB-224_224-lyr3.tar", map_location="cpu")

state_dict = checkpoint["model_state_dict"]

model.load_state_dict(state_dict, strict=True)
model.eval()

==> Add Sine PositionEmbedding~


/tmp/ipykernel_4693/556375509.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("./weights/TokenHPEv1-ViTB-224_224-lyr3.tar", map_location="cpu")


TokenHPE(
  (feature_extractor): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )

In [ ]:
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor()
    # T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

img_path = "dataset/300W_LP/AFW/AFW_134212_1_0.jpg"   # posa la que vulgues

img = Image.open(img_path).convert("RGB")
x = transform(img).unsqueeze(0)

with torch.no_grad():
    pred_rotation_matrix, ori_9_d = model(x)

print("Predicted Rotation Matrix Shape:", pred_rotation_matrix.shape)
print("Predicted Rotation Matrix:\n", pred_rotation_matrix[0])

Predicted Rotation Matrix Shape: torch.Size([1, 3, 3])
Predicted Rotation Matrix:
 tensor([[ 0.8834, -0.0207,  0.4681],
        [-0.1215,  0.9548,  0.2714],
        [-0.4525, -0.2967,  0.8410]])


In [ ]:
# %run inference.py --model_path ./weights/TokenHPEv1-ViTB-224_224-lyr3.tar --image_path dataset/300W_LP/AFW/AFW_134212_1_0.jpg --save_path output.png

In [13]:
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor()
    # T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

img_path = "dataset/300W_LP/AFW/AFW_134212_1_6.jpg"   

img = Image.open(img_path).convert("RGB")
x = transform(img).unsqueeze(0)

with torch.no_grad():
    pred_rotation_matrix, ori_9_d = model(x)

print("Predicted Rotation Matrix Shape:", pred_rotation_matrix.shape)
print("Predicted Rotation Matrix:\n", pred_rotation_matrix[0])

euler = utils.compute_euler_angles_from_rotation_matrices(pred_rotation_matrix) * 180 / np.pi
pitch = float(euler[0, 0])
yaw   = float(euler[0, 1])
roll  = float(euler[0, 2])

print("Pitch:", pitch, "Yaw:", yaw, "Roll:", roll)

cv_img = cv2.imread(img_path)

utils.draw_axis(cv_img, yaw, pitch, roll, size=100)

cv2.imshow("Pose", cv_img)
cv2.waitKey(0)
cv2.destroyAllWindows()


Predicted Rotation Matrix Shape: torch.Size([1, 3, 3])
Predicted Rotation Matrix:
 tensor([[ 0.5858, -0.0957,  0.8048],
        [-0.0781,  0.9817,  0.1736],
        [-0.8067, -0.1646,  0.5676]])
Pitch: -16.16709327697754 Yaw: 53.772430419921875 Roll: -7.592999458312988


# Model2 (SixDRepNet)

In [ ]:
# Create model
# Weights are automatically downloaded
model_sixD = SixDRepNet()

img = cv2.imread('dataset/300W_LP/AFW/AFW_134212_1_6.jpg')

pitch, yaw, roll = model_sixD.predict(img)

model_sixD.draw_axis(img, yaw, pitch, roll)

cv2.imshow("test_window", img)
cv2.waitKey(0)

Downloading: "https://cloud.ovgu.de/s/Q67RnLDy6JKLRWm/download/6DRepNet_300W_LP_AFLW2000.pth" to /home/anton/.cache/torch/hub/checkpoints/6DRepNet_300W_LP_AFLW2000.pth
100%|██████████| 150M/150M [00:38<00:00, 4.11MB/s] 


27